# Đánh giá OCR: PaddleOCR, VietOCR, Tesseract

Chạy các model OCR lên golden set do `src/data-golden-set-generator` sinh ra, chấm với `ground_truth.json` theo cùng một cách để so trực tiếp:

- **End-to-end:** OCR cả trang (tìm dòng + đọc), mỗi trường có đọc đúng không, theo mức nhiễu `clean → light → medium → heavy`.
- **Recognition-only:** cắt đúng bbox ground truth rồi đọc, tách lỗi nhận dạng khỏi lỗi tìm dòng.

| Chỉ số | Ý nghĩa | Tốt khi |
|---|---|---|
| CER / WER | Khoảng cách Levenshtein theo ký tự / theo từ, chia độ dài ground truth. End-to-end khớp theo đoạn con, vì dòng OCR thường dính nhãn (`Họ và tên: ...`) | Thấp |
| Field Accuracy | Tỷ lệ trường đọc đúng hoàn toàn | Cao |
| Document Accuracy | Tỷ lệ giấy tờ (ở một mức nhiễu) có **mọi** trường đúng | Cao |
| CER bỏ dấu | CER sau khi bỏ dấu và chữ hoa. Chênh với CER là phần lỗi do dấu | Thấp |
| Detection recall | Tỷ lệ trường có dòng OCR chồng lên | Cao |
| Độ suy giảm | CER(heavy) − CER(clean) | Thấp |
| ECE | Lệch giữa điểm tự tin và độ đúng thực tế (rec-only) | Thấp |

Trường `checkbox` bị loại. Dữ liệu tổng hợp nên metric đẹp hơn hồ sơ thật: dùng để so model, không dùng làm con số cuối.

## 0. Cài đặt

Local: dùng `jupyters/.venv`, xem `README.md`. Colab: chọn runtime GPU, upload và giải nén dữ liệu vào `/content/data` rồi chạy cell dưới.

In [ ]:
import os, sys
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ  # Kaggle: bật Internet trong Settings để pip cài được
if "google.colab" in sys.modules or ON_KAGGLE:
    !apt-get -qq install -y tesseract-ocr tesseract-ocr-vie > /dev/null
    import shutil
    if shutil.which("nvidia-smi"):       # runtime GPU
        %pip install -q paddlepaddle-gpu==3.3.1 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
    else:                                # runtime CPU: bản gpu cần libcuda.so.1, import sẽ lỗi
        %pip uninstall -q -y paddlepaddle-gpu
        %pip install -q paddlepaddle==3.3.1 -i https://www.paddlepaddle.org.cn/packages/stable/cpu/
    %pip install -q paddleocr==3.7.0 rapidfuzz pytesseract einops gdown prefetch_generator
    %pip install -q --no-deps vietocr==0.3.13
    # %pip lỗi mạng chỉ in WARNING rồi chạy tiếp: dừng ở đây cho rõ nguyên nhân
    import importlib.util
    if importlib.util.find_spec("rapidfuzz") is None:
        raise RuntimeError("pip install thất bại. Kaggle: bật Settings > Internet (cần xác minh số điện thoại), rồi Restart & Run All")

## 1. Cấu hình

`MODELS` có dạng `<engine>/<model>`. Bỏ dòng nào thì không chạy model đó. Mỗi engine chỉ import thư viện khi có model của nó trong danh sách.

In [ ]:
import os
os.environ.setdefault("PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK", "True")

import json, re, shutil, time, unicodedata, urllib.request
from collections import Counter
from functools import cache
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from rapidfuzz.distance import Levenshtein

KAGGLE_DATA = Path("/kaggle/input/datasets/balebomm/dataset/generated")  # dataset Kaggle, chỉ đọc
DATA_DIR = KAGGLE_DATA if KAGGLE_DATA.is_dir() else Path("data")  # thư mục generated/ của generator (Colab: /content/data)
OUTPUT_DIR = Path("outputs")             # cache OCR: outputs/<engine>/<model>/, chạy lại không OCR lại
# <doc-id>_p<N>.png (sạch), <doc-id>_p<N>_<level>.jpg (nhiễu). Ảnh *_bbox.jpg không khớp
IMAGE_PATTERN = re.compile(r"^(?P<doc_id>.+)_p(?P<page>\d+)(?:\.png|_(?P<level>light|medium|heavy)\.jpg)$")
MAX_IMAGES = 24                          # None = tất cả. CPU: PaddleOCR ~25s, VietOCR ~10s, Tesseract vài giây mỗi ảnh/model
LEVELS = ["clean", "light", "medium", "heavy"]

MODELS = [
    "paddleocr/PP-OCRv6_medium_rec",     # model PaddleOCR chọn cho lang="vi"
    "paddleocr/latin_PP-OCRv5_mobile_rec",
    "paddleocr/PP-OCRv5_server_rec",
    "vietocr/vgg-transformer",           # model mặc định của VietOCR
    "vietocr/vgg-seq2seq",               # nhanh hơn ~3 lần
    "tesseract/vie-apt-psm3",            # gói vie của apt, tự phân tích bố cục
    "tesseract/vie-best-psm3",           # tessdata_best
    "tesseract/vie-best-psm11",          # tessdata_best, chữ rời rạc (hợp biểu mẫu)
]
ENGINE_NAMES = list(dict.fromkeys(m.split("/")[0] for m in MODELS))

# PaddleOCR và VietOCR dùng chung detection này, với tham số của pipeline OCR PaddleOCR
# (paddlex/configs/pipelines/OCR.yaml). Module TextDetection mặc định khác, nên ghi rõ.
DET_MODEL = "PP-OCRv6_medium_det"
DET_PARAMS = dict(limit_side_len=64, limit_type="min", thresh=0.3, box_thresh=0.6, unclip_ratio=1.5)

VIETOCR_DIR = Path("models/vietocr")     # vocr.vn đã ngừng, tải config từ GitHub và trọng số từ release v0.3.2
VIETOCR_CONFIG_URL = "https://raw.githubusercontent.com/pbcquoc/vietocr/master/config/"
VIETOCR_WEIGHTS_URL = "https://github.com/pbcquoc/vietocr/releases/download/v0.3.2/"
TESSDATA_BEST = Path("models/tessdata_best")
TESS_CONFIGS = {
    "vie-apt-psm3":   dict(lang="vie", tessdata=None, psm=3),
    "vie-best-psm3":  dict(lang="vie", tessdata=TESSDATA_BEST, psm=3),
    "vie-best-psm11": dict(lang="vie", tessdata=TESSDATA_BEST, psm=11),
}
pd.set_option("display.max_colwidth", 80)

In [ ]:
# Biểu đồ: màu theo engine, nét theo model trong engine
PALETTE = {"paddleocr": "#2a78d6", "vietocr": "#eb6834", "tesseract": "#1baf7a"}
DASHES = ["-", "--", ":"]
COLOR = {m: PALETTE[m.split("/")[0]] for m in MODELS}
STYLE = {m: DASHES[[x for x in MODELS if x.split("/")[0] == m.split("/")[0]].index(m) % 3] for m in MODELS}
SEQ = LinearSegmentedColormap.from_list("seq_blue", ["#cde2fb", "#86b6ef", "#3987e5", "#256abf", "#184f95", "#0d366b"])
INK, MUTED, GRID, SURFACE = "#1f2328", "#59636e", "#d8dee4", "#fcfcfb"
plt.rcParams.update({"figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
                     "axes.titlesize": 11, "axes.labelcolor": MUTED, "text.color": INK})
pct = plt.FuncFormatter(lambda v, _: f"{v * 100:.4g}%")

def style_axes(ax, grid="y", percent_axis="y"):
    ax.grid(axis=grid, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)
    ax.tick_params(colors=MUTED, length=0)
    if percent_axis:
        getattr(ax, f"{percent_axis}axis").set_major_formatter(pct)

def legend(fig):
    handles = [plt.Line2D([], [], color=COLOR[m], linestyle=STYLE[m], linewidth=2, marker="o", markersize=7,
                          markeredgecolor="white") for m in MODELS]
    fig.legend(handles, MODELS, frameon=False, ncol=min(len(MODELS), 3), loc="lower center", bbox_to_anchor=(0.5, -0.1))

## 2. Dữ liệu và ground truth

Quét đệ quy `DATA_DIR` lấy ảnh khớp `IMAGE_PATTERN`, tra ground truth từ mọi `ground_truth.json`. Mỗi dòng của `items` là một trường trên một ảnh. Bbox ảnh nhiễu đã được generator biến đổi theo phép xoay/phối cảnh.

- **Local:** `ln -s ../src/data-golden-set-generator/generated data`
- **Colab:** zip nội dung bên trong `generated/` thành `generated.zip`, upload vào `/content`. Cell dưới tự giải nén vào `data/` nếu chưa có
- **Kaggle:** add dataset `balebomm/dataset`, notebook tự dùng `/kaggle/input/datasets/balebomm/dataset/generated`. Kết quả ghi vào `outputs/` trong `/kaggle/working`

In [ ]:
DATA_ZIP = Path("generated.zip")          # nội dung bên trong generated/, upload cạnh notebook (Colab: /content)
if not DATA_DIR.is_dir() and DATA_ZIP.exists():
    import zipfile
    print(f"Giải nén {DATA_ZIP} -> {DATA_DIR}/")
    with zipfile.ZipFile(DATA_ZIP) as z:
        z.extractall(DATA_DIR)
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Không thấy {DATA_DIR.resolve()} và {DATA_ZIP.resolve()}. Xem hướng dẫn đặt dữ liệu ở trên.")

images = sorted(p.as_posix() for p in DATA_DIR.rglob("*") if IMAGE_PATTERN.match(p.name))[:MAX_IMAGES]

def index_ground_truth(data_dir: Path) -> dict:
    """Đường dẫn ảnh -> (thông tin giấy tờ, level, page, bbox theo tên trường)."""
    index = {}
    for gt_path in data_dir.rglob("ground_truth.json"):
        gt = json.loads(gt_path.read_text(encoding="utf-8"))
        for doc in gt["documents"]:
            clean_boxes = {f["name"]: f.get("bbox", []) for f in doc["fields"]}
            pages = [("clean", i + 1, p, clean_boxes) for i, p in enumerate(doc["files"]["pages"])]
            pages += [(n["level"], n["page"], n["path"], n["bboxes"]) for n in doc["files"].get("noisy", [])]
            for level, page, rel, boxes in pages:
                index[(gt_path.parent / rel).as_posix()] = (gt["dossier_id"], doc, level, page, boxes)
    return index

def group_lines(boxes):
    """Gom các bbox nằm ở dòng kế tiếp nhau (trường dài bị xuống dòng) thành một lần xuất hiện.
    Bbox cách xa nhau (trường in ở nhiều chỗ, ví dụ tên ở đầu trang và chỗ ký) vẫn tách riêng."""
    groups = []
    for b in sorted(boxes, key=lambda b: (b["y"], b["x"])):
        if groups:
            p = groups[-1][-1]
            next_line = b["y"] > p["y"] + p["h"] * 0.5 and b["y"] - (p["y"] + p["h"]) < p["h"] * 0.8
            if next_line:
                groups[-1].append(b)
                continue
        groups.append([b])
    return groups

def build_items(images, gt_index) -> pd.DataFrame:
    rows = []
    for im in images:
        if im not in gt_index:
            continue
        dossier, doc, level, page, boxes = gt_index[im]
        for f in doc["fields"]:
            if f.get("fill") == "checkbox" or not f["text"].strip():
                continue
            page_boxes = [bb for bb in boxes.get(f["name"], []) if bb.get("page", 1) == page]
            for k, group in enumerate(group_lines(page_boxes)):
                x0, y0 = min(b["x"] for b in group), min(b["y"] for b in group)
                x1, y1 = max(b["x"] + b["w"] for b in group), max(b["y"] + b["h"] for b in group)
                rows.append(dict(
                    dossier=dossier, doc_id=doc["doc_id"], doc_type=doc["doc_type"], level=level, image=im,
                    field=f["name"], field_type=f["type"], fill=f.get("fill", "printed"),
                    occurrence=k, truth=f["text"], n_lines=len(group),
                    boxes=[(b["x"], b["y"], b["w"], b["h"]) for b in group],
                    x=x0, y=y0, w=x1 - x0, h=y1 - y0,     # khung bao, dùng để cắt ảnh xem
                ))
    return pd.DataFrame(rows)

gt_index = index_ground_truth(DATA_DIR)
items = build_items(images, gt_index)
missing = [im for im in images if im not in gt_index]
if missing:
    print(f"Bỏ qua {len(missing)} ảnh không có trong ground_truth.json nào, ví dụ {missing[0]}")
    images = [im for im in images if im in gt_index]
if items.empty:
    raise RuntimeError(f"Không tìm thấy ảnh generator trong {DATA_DIR.resolve()}")
print(f"{len(images)} ảnh, {items.dossier.nunique()} hồ sơ, {len(items)} trường cần đọc")
items["level"] = pd.Categorical(items["level"], LEVELS, ordered=True)
items.groupby(["doc_type", "fill"]).field.unique().to_frame("fields")

## 3. Engine

Mỗi engine có 2 hàm tạo, nhận tên model và trả về hàm đọc:

- `page(name)` → `f(image_path)`: OCR cả trang, trả danh sách dòng `{text, score, box=[x0, y0, x1, y1]}`.
- `line(name)` → `f(img_bgr)`: đọc một vùng cắt một dòng, trả `(text, score)`.

| Engine | Tìm dòng | Đọc | `score` |
|---|---|---|---|
| PaddleOCR | `DET_MODEL` | model rec của PaddleOCR | điểm rec |
| VietOCR | `DET_MODEL`, cắt thẳng theo đa giác | VietOCR (theo lô) | xác suất TB các ký tự |
| Tesseract | tự tìm, gom từ theo `(block, par, line)` | LSTM (`--oem 1`), rec-only dùng `--psm 7` | độ tự tin TB các từ |

Thang `score` mỗi engine một khác: chỉ so ECE, không so trực tiếp điểm.

In [ ]:
@cache
def paddle_kwargs():
    import paddle
    if paddle.device.is_compiled_with_cuda() and paddle.device.cuda.device_count() > 0:
        return dict(device="gpu:0")
    return dict(device="cpu", enable_mkldnn=False)   # Paddle CPU 3.x lỗi oneDNN với model v6

# --- PaddleOCR
def paddle_page(name):
    from paddleocr import PaddleOCR
    ocr = PaddleOCR(text_detection_model_name=DET_MODEL, text_recognition_model_name=name,
                    use_doc_orientation_classify=False, use_doc_unwarping=False, use_textline_orientation=False,
                    **{f"text_det_{k}": v for k, v in DET_PARAMS.items()}, **paddle_kwargs())
    def run(path):
        r = ocr.predict(path)[0]
        return [dict(text=t, score=float(s), box=[int(v) for v in b])
                for t, s, b in zip(r["rec_texts"], r["rec_scores"], r["rec_boxes"])]
    return run

def paddle_line(name):
    from paddleocr import TextRecognition
    rec = TextRecognition(model_name=name, **paddle_kwargs())
    def run(img):
        r = rec.predict(img)[0]
        return r["rec_text"], float(r["rec_score"])
    return run

# --- VietOCR: chỉ đọc một dòng, dùng detection của PaddleOCR để tìm dòng
@cache
def detector():
    from paddleocr import TextDetection
    return TextDetection(model_name=DET_MODEL, **DET_PARAMS, **paddle_kwargs())

def _fetch(url, dst):
    if not dst.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(url, dst)
    return dst

@cache
def vietocr(name):
    import torch, yaml
    from vietocr.tool.config import Cfg
    from vietocr.tool.predictor import Predictor
    cfg = {}
    for f in ("base", name):
        cfg.update(yaml.safe_load(_fetch(VIETOCR_CONFIG_URL + f"{f}.yml", VIETOCR_DIR / f"{f}.yml").read_text(encoding="utf-8")))
    cfg = Cfg(cfg)
    cfg["weights"] = str(_fetch(VIETOCR_WEIGHTS_URL + f"{name}.pth", VIETOCR_DIR / f"{name}.pth"))
    cfg["device"] = "cuda:0" if torch.cuda.is_available() else "cpu"
    cfg["cnn"]["pretrained"] = False
    cfg["predictor"]["beamsearch"] = False
    return Predictor(cfg)

def crop_poly(img, poly):
    """Cắt thẳng một dòng theo đa giác 4 điểm (trái trên, phải trên, phải dưới, trái dưới)."""
    pts = np.asarray(poly, dtype=np.float32)
    w = int(max(np.linalg.norm(pts[0] - pts[1]), np.linalg.norm(pts[3] - pts[2])))
    h = int(max(np.linalg.norm(pts[0] - pts[3]), np.linalg.norm(pts[1] - pts[2])))
    M = cv2.getPerspectiveTransform(pts, np.float32([[0, 0], [w, 0], [w, h], [0, h]]))
    out = cv2.warpPerspective(img, M, (max(w, 1), max(h, 1)), borderMode=cv2.BORDER_REPLICATE)
    return np.rot90(out) if h >= 1.5 * w else out     # dòng dọc: xoay nằm ngang

def to_pil(img_bgr):
    from PIL import Image
    return Image.fromarray(cv2.cvtColor(np.ascontiguousarray(img_bgr), cv2.COLOR_BGR2RGB))

def vietocr_page(name):
    det, pred = detector(), vietocr(name)
    def run(path):
        img = cv2.imread(path)
        polys = np.asarray(det.predict(path)[0]["dt_polys"])
        if len(polys) == 0:
            return []
        texts, probs = pred.predict_batch([to_pil(crop_poly(img, p)) for p in polys], return_prob=True)
        return [dict(text=t, score=float(s), box=[int(p[:, 0].min()), int(p[:, 1].min()), int(p[:, 0].max()), int(p[:, 1].max())])
                for t, s, p in zip(texts, probs, polys)]
    return run

def vietocr_line(name):
    pred = vietocr(name)
    def run(img):
        text, prob = pred.predict(to_pil(img), return_prob=True)
        return text, float(prob)
    return run

# --- Tesseract: chương trình ngoài, gọi qua pytesseract
LINE_BORDER = 10                          # viền trắng khi đọc vùng cắt: Tesseract đọc kém khi chữ sát mép

@cache
def tesseract_ready():
    if shutil.which("tesseract") is None:
        raise RuntimeError("Chưa cài tesseract: sudo apt install tesseract-ocr tesseract-ocr-vie")
    _fetch("https://github.com/tesseract-ocr/tessdata_best/raw/main/vie.traineddata", TESSDATA_BEST / "vie.traineddata")
    return True

def _words(img_rgb, cfg, psm):
    import pytesseract
    tesseract_ready()
    config = f"--oem 1 --psm {psm}"
    if cfg["tessdata"] is not None:
        config += f' --tessdata-dir "{Path(cfg["tessdata"]).resolve()}"'
    d = pytesseract.image_to_data(img_rgb, lang=cfg["lang"], config=config, output_type=pytesseract.Output.DICT)
    for i, word in enumerate(d["text"]):
        conf = float(d["conf"][i])
        if word.strip() and conf >= 0:
            yield d, i, word, conf / 100

def tesseract_page(name):
    cfg = TESS_CONFIGS[name]
    def run(path):
        img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        lines = {}
        for d, i, word, conf in _words(img, cfg, cfg["psm"]):
            x, y, w, h = d["left"][i], d["top"][i], d["width"][i], d["height"][i]
            ln = lines.setdefault((d["block_num"][i], d["par_num"][i], d["line_num"][i]), dict(words=[], confs=[], box=[x, y, x + w, y + h]))
            ln["words"].append(word)
            ln["confs"].append(conf)
            b = ln["box"]
            ln["box"] = [min(b[0], x), min(b[1], y), max(b[2], x + w), max(b[3], y + h)]
        return [dict(text=" ".join(l["words"]), score=float(np.mean(l["confs"])), box=[int(v) for v in l["box"]])
                for l in lines.values()]
    return run

def tesseract_line(name):
    cfg = TESS_CONFIGS[name]
    def run(img):
        img = cv2.copyMakeBorder(img, *[LINE_BORDER] * 4, cv2.BORDER_CONSTANT, value=(255, 255, 255))
        words = [(w, c) for _, _, w, c in _words(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), cfg, 7)]
        return (" ".join(w for w, _ in words), float(np.mean([c for _, c in words]))) if words else ("", 0.0)
    return run

ENGINES = {"paddleocr": (paddle_page, paddle_line), "vietocr": (vietocr_page, vietocr_line),
           "tesseract": (tesseract_page, tesseract_line)}

## 4. OCR cả trang

Kết quả mỗi ảnh cache thành JSON trong `outputs/<engine>/<model>/`. Xóa thư mục để chạy lại.

In [ ]:
def cache_path(model, image):
    return OUTPUT_DIR / model / Path(image).relative_to(DATA_DIR).with_suffix(".json")

def run_pages(model):
    todo = [im for im in images if not cache_path(model, im).exists()]
    print(f"{model}: {len(images) - len(todo)} ảnh đã cache, chạy {len(todo)} ảnh")
    if not todo:
        return
    engine, name = model.split("/", 1)
    ocr = ENGINES[engine][0](name)
    for i, im in enumerate(todo, 1):
        t0 = time.perf_counter()
        out = dict(lines=ocr(im))
        out["seconds"] = time.perf_counter() - t0
        cp = cache_path(model, im)
        cp.parent.mkdir(parents=True, exist_ok=True)
        cp.write_text(json.dumps(out, ensure_ascii=False), encoding="utf-8")
        if i % 10 == 0 or i == len(todo):
            print(f"  {i}/{len(todo)} ({out['seconds']:.1f}s/ảnh)")

for m in MODELS:
    run_pages(m)

## 5. Chấm điểm

End-to-end: gộp các dòng OCR chồng lên bbox trường rồi tìm đoạn con khớp nhất. Rec-only: cắt từng dòng của trường theo bbox ground truth (nới `PAD` px), đọc rồi nối lại, so toàn chuỗi.

In [ ]:
def norm(s: str) -> str:
    return " ".join(unicodedata.normalize("NFC", s).split())

def strip_marks(s: str) -> str:
    s = unicodedata.normalize("NFD", s.replace("đ", "d").replace("Đ", "D"))
    return "".join(c for c in s if unicodedata.category(c) != "Mn").casefold()

def semiglobal_distance(gt: str, hyp: str) -> int:
    """Số phép sửa ít nhất để gt thành một đoạn con liên tiếp của hyp. Dùng được cho chuỗi ký tự hoặc danh sách từ."""
    if not gt:
        return 0
    prev = [0] * (len(hyp) + 1)
    for i, g in enumerate(gt, 1):
        cur = [i] + [0] * len(hyp)
        for j, h in enumerate(hyp, 1):
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (g != h))
        prev = cur
    return min(prev)

def overlap_lines(lines, x, y, w, h, min_cover=0.5):
    """Dòng OCR mà phần giao chiếm >= min_cover diện tích của bên nhỏ hơn (trường hoặc dòng)."""
    hits = []
    for ln in lines:
        x0, y0, x1, y1 = ln["box"]
        iw = min(x + w, x1) - max(x, x0)
        ih = min(y + h, y1) - max(y, y0)
        if iw <= 0 or ih <= 0:
            continue
        if iw * ih / min(w * h, (x1 - x0) * (y1 - y0)) >= min_cover:
            hits.append(ln)
    return sorted(hits, key=lambda l: ((l["box"][1] + l["box"][3]) / 2, l["box"][0]))

def score_e2e(model):
    pages = {im: json.loads(cache_path(model, im).read_text(encoding="utf-8")) for im in items.image.unique()}
    out = []
    for r in items.itertuples():
        found = {id(ln): ln for box in r.boxes for ln in overlap_lines(pages[r.image]["lines"], *box)}
        hits = sorted(found.values(), key=lambda l: ((l["box"][1] + l["box"][3]) / 2, l["box"][0]))
        hyp, gt = norm(" ".join(l["text"] for l in hits)), norm(r.truth)
        out.append(dict(
            detected=bool(hits), hyp=hyp,
            score=float(np.mean([l["score"] for l in hits])) if hits else 0.0,
            cer=min(semiglobal_distance(gt, hyp) / len(gt), 1),
            cer_nomark=min(semiglobal_distance(strip_marks(gt), strip_marks(hyp)) / len(gt), 1),
            wer=min(semiglobal_distance(gt.split(), hyp.split()) / len(gt.split()), 1),
        ))
    return items.assign(model=model, **pd.DataFrame(out, index=items.index))

PAD = 6

def crop_box(img, x, y, w, h):
    H, W = img.shape[:2]
    return img[max(int(y) - PAD, 0):min(int(y + h) + PAD, H), max(int(x) - PAD, 0):min(int(x + w) + PAD, W)]

def score_rec(model):
    cp = OUTPUT_DIR / model / "_rec_only.json"
    cache_ = json.loads(cp.read_text(encoding="utf-8")) if cp.exists() else {}
    keys = [f"{Path(r.image).relative_to(DATA_DIR).as_posix()}|{r.field}|{r.occurrence}" for r in items.itertuples()]
    todo = [(k, r) for k, r in zip(keys, items.itertuples()) if k not in cache_]
    if todo:
        print(f"{model}: rec-only {len(todo)} vùng cắt")
        engine, name = model.split("/", 1)
        read, imgs = ENGINES[engine][1](name), {}
        for k, r in todo:
            img = imgs.setdefault(r.image, cv2.imread(r.image))
            res = [read(crop_box(img, *box)) for box in r.boxes]
            cache_[k] = dict(text=" ".join(t for t, _ in res), score=float(np.mean([s for _, s in res])))
        cp.parent.mkdir(parents=True, exist_ok=True)
        cp.write_text(json.dumps(cache_, ensure_ascii=False), encoding="utf-8")
    hyp, gt = [norm(cache_[k]["text"]) for k in keys], [norm(g) for g in items.truth]
    lev = lambda a, b: min(Levenshtein.distance(a, b) / len(a), 1)
    return items.assign(
        model=model, hyp=hyp, score=[cache_[k]["score"] for k in keys],
        cer=[lev(g, h) for g, h in zip(gt, hyp)],
        cer_nomark=[lev(strip_marks(g), strip_marks(h)) for g, h in zip(gt, hyp)],
        wer=[lev(g.split(), h.split()) for g, h in zip(gt, hyp)],
    )

e2e = pd.concat([score_e2e(m) for m in MODELS], ignore_index=True)
rec = pd.concat([score_rec(m) for m in MODELS], ignore_index=True)
for df in (e2e, rec):
    df["exact"] = df.cer == 0
seconds = pd.Series({m: np.mean([json.loads(cache_path(m, im).read_text())["seconds"] for im in images]) for m in MODELS})

for name, df in [("e2e", e2e), ("rec", rec)]:     # dataset-profile.ipynb đọc để vẽ benchmark theo loại giấy tờ
    df.drop(columns="boxes").to_csv(OUTPUT_DIR / f"scores_{name}.csv", index=False)

## 6. Kết quả

### 6.1. Bảng tổng hợp

Đậm là giá trị tốt nhất mỗi cột. Giây/ảnh chỉ so được giữa các model chạy cùng thiết bị.

In [ ]:
MAIN = [("CER", False), ("WER", False), ("Field Accuracy", True), ("Document Accuracy", True)]

def main_metrics(df, by=()):
    keys = ["model", *by]
    fields = df.groupby(keys, observed=True).agg(**{"CER": ("cer", "mean"), "WER": ("wer", "mean"),
                                                    "Field Accuracy": ("exact", "mean")})
    docs = (df.groupby(["model", "dossier", "doc_id", "level", *[b for b in by if b != "level"]], observed=True)
              .exact.all().groupby(keys, observed=True).mean().rename("Document Accuracy"))
    return fields.join(docs)

def calibration(df, bins=(0, 0.5, 0.7, 0.8, 0.9, 0.95, 0.98, 1.0001)):
    g = df.groupby(pd.cut(df.score, list(bins), right=False), observed=True).agg(
        conf=("score", "mean"), acc=("exact", "mean"), n=("exact", "size"))
    return g, float((g.n / g.n.sum() * (g.acc - g.conf).abs()).sum())

main_e2e, main_rec = main_metrics(e2e), main_metrics(rec)
cer_lv = e2e.groupby(["model", "level"], observed=True).cer.mean().unstack("level").reindex(columns=LEVELS)
coef = main_e2e.join(main_rec.add_suffix(" (rec-only)")).assign(**{
    "CER bỏ dấu": e2e.groupby("model").cer_nomark.mean(),
    "Detection recall": e2e.groupby("model").detected.mean(),
    "Độ suy giảm": cer_lv["heavy"] - cer_lv["clean"],
    "ECE": pd.Series({m: calibration(rec[rec.model == m])[1] for m in MODELS}),
    "Giây/ảnh": seconds,
}).loc[MODELS]
higher = [c for c in coef.columns if "Accuracy" in c or c == "Detection recall"]
(coef.style
    .format({c: "{:.1%}" for c in coef.columns if c != "Giây/ảnh"} | {"Giây/ảnh": "{:.1f}"})
    .highlight_min(subset=[c for c in coef.columns if c not in higher], props="font-weight: bold")
    .highlight_max(subset=higher, props="font-weight: bold"))

### 6.2. Bốn chỉ số chính

In [ ]:
def scorecard(table, title):
    fig, axes = plt.subplots(1, len(MAIN), figsize=(3.2 * len(MAIN) + 2.5, 0.42 * len(MODELS) + 1.4), sharey=True)
    y = np.arange(len(MODELS))
    for i, (ax, (col, hb)) in enumerate(zip(axes, MAIN)):
        vals = table[col].reindex(MODELS).values
        ax.barh(y, vals, height=0.62, color=[COLOR[m] for m in MODELS], edgecolor=SURFACE, linewidth=2)
        for yi, v in zip(y, vals):
            ax.text(v, yi, f" {v:.1%}", va="center", fontsize=9, color=INK)
        ax.set_xlim(0, 1.25 if hb else max(np.nanmax(vals) * 1.45, 0.05))
        ax.set_title(f"{col}\n({'cao' if hb else 'thấp'} là tốt)", loc="left", fontsize=10)
        style_axes(ax, grid="x", percent_axis="x")
        ax.tick_params(labelleft=(i == 0))
    axes[0].set_yticks(y, MODELS)
    axes[0].invert_yaxis()
    fig.suptitle(title, x=0.01, ha="left")
    plt.tight_layout()

scorecard(main_e2e, "End-to-end (tìm dòng + đọc, cả trang)")
scorecard(main_rec, "Recognition-only (vùng cắt theo bbox ground truth)")

### 6.3. Theo mức nhiễu

End-to-end. Đường dốc mạnh là model nhạy với chất lượng ảnh. Document Accuracy rơi nhanh nhất vì một trường sai là cả giấy tờ trượt.

In [ ]:
by_level = main_metrics(e2e, by=["level"])
fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))
for ax, (col, hb) in zip(axes, MAIN):
    for m in MODELS:
        s = by_level.loc[m, col].reindex(LEVELS)
        ax.plot(LEVELS, s.values, color=COLOR[m], linestyle=STYLE[m], linewidth=2, marker="o", markersize=7,
                markeredgecolor="white", markeredgewidth=2)
    ax.set_ylim(0, 1.05 if hb else max(by_level[col].max() * 1.2, 0.05))
    ax.set_title(f"{col} ({'cao' if hb else 'thấp'} là tốt)", loc="left")
    style_axes(ax)
plt.tight_layout(rect=(0, 0.12, 1, 1))
legend(fig)

### 6.4. Nguồn lỗi: mặt chữ hay dấu

**Lỗi mặt chữ** = CER bỏ dấu (sai cả khi bỏ dấu). **Lỗi dấu** = CER − CER bỏ dấu. Lỗi dấu lớn là model đọc đúng chữ cái nhưng chưa hợp tiếng Việt.

In [ ]:
rows = []
for mode, df in [("end-to-end", e2e), ("rec-only", rec)]:
    g = df.groupby("model")[["cer", "cer_nomark"]].mean()
    rows += [dict(model=m, mode=mode, face=g.loc[m, "cer_nomark"], mark=max(g.loc[m, "cer"] - g.loc[m, "cer_nomark"], 0)) for m in MODELS]
err = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(13, 0.42 * len(MODELS) + 1.4), sharey=True)
y = np.arange(len(MODELS))
for ax, mode in zip(axes, ["end-to-end", "rec-only"]):
    d = err[err["mode"] == mode].set_index("model").loc[MODELS]
    ax.barh(y, d.face, height=0.62, color="#184f95", edgecolor=SURFACE, linewidth=2, label="Lỗi mặt chữ")
    ax.barh(y, d.mark, left=d.face, height=0.62, color="#86b6ef", edgecolor=SURFACE, linewidth=2, label="Lỗi dấu")
    for yi, t in zip(y, d.face + d.mark):
        ax.text(t, yi, f" {t:.1%}", va="center", fontsize=9)
    ax.set_xlim(0, max((err.face + err.mark).max() * 1.25, 0.05))
    ax.set_title(f"CER {mode}", loc="left")
    style_axes(ax, grid="x", percent_axis="x")
axes[0].set_yticks(y, MODELS)
axes[0].invert_yaxis()
axes[1].legend(frameon=False, ncol=2, loc="lower right", bbox_to_anchor=(1, 1))
plt.tight_layout()

### 6.5. Theo kiểu điền và theo trường

CER end-to-end theo kiểu điền (viết tay / đánh máy / in sẵn) × mức nhiễu, và `TOP_FIELDS` trường khó nhất, kèm model đọc tốt nhất.

In [ ]:
(e2e.pivot_table(index=["fill", "level"], columns="model", values="cer", aggfunc="mean", observed=True)[MODELS]
    .style.format("{:.1%}").background_gradient(cmap=SEQ, vmin=0, vmax=0.5))

In [ ]:
TOP_FIELDS = 20
by_field = e2e.pivot_table(index=["doc_type", "field", "fill"], columns="model", values="cer", aggfunc="mean", observed=True)[MODELS]
by_field = by_field.loc[by_field.mean(axis=1).sort_values(ascending=False).index[:TOP_FIELDS]]
(by_field.assign(**{"tốt nhất": by_field.idxmin(axis=1)})
    .style.format("{:.1%}", subset=MODELS).background_gradient(cmap=SEQ, vmin=0, vmax=0.5, subset=MODELS))

### 6.6. Hiệu chỉnh điểm tự tin

Rec-only. Model hiệu chỉnh tốt nằm sát đường chéo. Nằm dưới là tự tin quá mức: không dùng được ngưỡng điểm để chọn trường cần người kiểm lại.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.plot([0, 1], [0, 1], color=MUTED, linewidth=1, linestyle="--")
for m in MODELS:
    g, ece = calibration(rec[rec.model == m])
    ax.plot(g.conf, g.acc, color=COLOR[m], linestyle=STYLE[m], linewidth=2, marker="o", markersize=7,
            markeredgecolor="white", markeredgewidth=2, label=f"{m} (ECE {ece:.1%})")
ax.set(xlim=(0, 1.02), ylim=(0, 1.02), aspect="equal", xlabel="Điểm tự tin trung bình", ylabel="Field Accuracy thực tế")
ax.set_title("Reliability diagram, recognition-only", loc="left")
style_axes(ax, grid="both")
ax.xaxis.set_major_formatter(pct)
ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(0, -0.14), fontsize=8)
plt.tight_layout()

## 7. Soi lỗi

### 7.1. Ký tự hay sai

Rec-only, ảnh sạch. Mỗi model 6 ký tự có tỷ lệ sai cao nhất (xuất hiện ≥ 5 lần), dạng `ký tự tỷ lệ (→ bị đọc thành)`.

In [ ]:
def char_errors(df, top=6, min_seen=5):
    seen, wrong, became = Counter(), Counter(), {}
    for g, h in zip(df.truth.map(norm), df.hyp):
        seen.update(g)
        for op in Levenshtein.editops(g, h):
            if op.tag in ("delete", "replace"):
                c = g[op.src_pos]
                wrong[c] += 1
                became.setdefault(c, Counter())[h[op.dest_pos] if op.tag == "replace" else "∅"] += 1
    worst = sorted((c for c in wrong if c != " " and seen[c] >= min_seen), key=lambda c: -wrong[c] / seen[c])[:top]
    return [f"{c} {wrong[c] / seen[c]:.0%} (→{became[c].most_common(1)[0][0]})" for c in worst]

pd.DataFrame({m: pd.Series(char_errors(rec[(rec.model == m) & (rec.level == "clean")]), dtype=object) for m in MODELS}).T.fillna("")

### 7.2. Cùng một trường, mỗi model đọc ra gì

Ảnh sạch, end-to-end. Các trường mà các model chênh CER nhiều nhất.

In [ ]:
wide = e2e[e2e.level == "clean"].pivot_table(index=["image", "field", "occurrence", "truth"], columns="model",
                                             values=["hyp", "cer"], aggfunc="first")
spread = (wide["cer"].max(axis=1) - wide["cer"].min(axis=1)).sort_values(ascending=False)
wide.loc[spread.index[:15], "hyp"][MODELS].reset_index(["image", "occurrence"], drop=True)

### 7.3. Ca sai nặng nhất của một model

Đổi `WORST_MODEL`, `WORST_LEVEL` để xem model / mức nhiễu khác.

In [ ]:
WORST_MODEL, WORST_LEVEL, N_WORST = MODELS[0], "clean", 10

s = rec[(rec.model == WORST_MODEL) & (rec.level == WORST_LEVEL) & (rec.cer > 0)].sort_values("cer", ascending=False).head(N_WORST)
if s.empty:
    print("Không có ca sai")
else:
    fig, axes = plt.subplots(len(s), 1, figsize=(10, 1.1 * len(s)))
    for ax, r in zip(np.atleast_1d(axes), s.itertuples()):
        ax.imshow(cv2.cvtColor(crop_box(cv2.imread(r.image), r.x, r.y, r.w, r.h), cv2.COLOR_BGR2RGB))
        ax.axis("off")
        ax.set_title(f"[{r.level}/{r.fill}] {r.field}  CER {r.cer:.0%}\nGT : {r.truth}\nOCR: {r.hyp}",
                     loc="left", fontsize=9, family="monospace")
    plt.tight_layout()

## 8. Ghi chú đọc kết quả

- **CER end-to-end cao hơn rec-only nhiều:** lỗi ở khâu tìm dòng (dòng bị cắt đôi, dính nhau) hoặc ngưỡng `min_cover` của `overlap_lines`. PaddleOCR và VietOCR dùng chung detection, nên chênh lệch end-to-end giữa hai engine là do phần đọc.
- **CER bỏ dấu thấp nhưng CER cao:** đọc được mặt chữ, hỏng ở dấu, cần model có đủ ký tự dấu chồng tiếng Việt.
- **VietOCR** đưa dòng về cao 32px, rộng tối đa 512px, nên dòng dài (nhãn + giá trị) bị ép ngang, hay mất hoặc thêm chữ ở cuối dòng. Nâng `image_max_width` còn tệ hơn vì model được train với ảnh ≤ 512px. Hướng sửa: cắt dòng dài tại khoảng trắng trước khi đọc.
- **Tesseract** detection recall thấp: không tìm ra dòng, hay gặp với chữ viết tay, ảnh nghiêng, nền bẩn. Thử psm khác hoặc tiền xử lý (xoay thẳng, nhị phân hóa).
- Chữ viết tay trong golden set là font viết tay, dễ hơn chữ viết tay thật. Cần kiểm lại trên `real/` trước khi chốt model.